In [ ]:
#!pip install requests pandas tqdm

import requests
import pandas as pd
from tqdm import tqdm
import time
from datetime import datetime
from pathlib import Path

# Defina a pasta base onde os arquivos serão salvos
BASE_DIR = Path.home() / "/Users/nathaliasales/Dropbox/MiDES-data-paper-replication/Data/Raw/comprasdados"
BASE_DIR.mkdir(parents=True, exist_ok=True)  # garante que a pasta existe

# Função para requisição com retry em caso de 429
def requisicao_com_retry(url, params, tentativas=3, espera=5):
    for i in range(tentativas):
        resp = requests.get(url, params=params)
        if resp.status_code == 200:
            return resp
        elif resp.status_code == 429:
            print(f"Recebido 429, esperando {espera} segundos... (tentativa {i+1})")
            time.sleep(espera)
        else:
            print(f"Erro na requisição: {resp.status_code}")
            print("Conteúdo da resposta:", resp.text[:500])
            return None
    print("Falha após múltiplas tentativas.")
    return None

# Função para extrair licitações de um período
def extrair_licitacoes(data_inicial, data_final, save_path, page_size=500, sleep_between=1):
    url_base = "https://dadosabertos.compras.gov.br/modulo-legado/1_consultarLicitacao"
    pagina = 1
    total_paginas = None
    total_saved = 0
    first_write = True

    pbar = tqdm(desc=f"{data_inicial} → {data_final}", unit="pág", dynamic_ncols=True)

    while True:
        params = {
            "pagina": pagina,
            "tamanhoPagina": page_size,
            "data_publicacao_inicial": data_inicial,
            "data_publicacao_final": data_final
        }

        resp = requisicao_com_retry(url_base, params)
        if not resp:
            break

        try:
            dados = resp.json()
        except ValueError:
            print("Resposta não é JSON — fim da extração.")
            break

        if isinstance(dados, dict) and "resultado" in dados:
            resultados = dados.get("resultado", [])
            if total_paginas is None:
                total_paginas = dados.get("totalPaginas", 1)
                print(f"Total de páginas esperado: {total_paginas}")
        else:
            resultados = dados

        if not resultados:
            print(f"Nenhum dado novo na página {pagina} — encerrando loop.")
            break

        df = pd.DataFrame(resultados)
        if not df.empty:
            df.to_csv(save_path, mode="w" if first_write else "a",
                      header=first_write, index=False, encoding="utf-8-sig")
            first_write = False
            total_saved += len(df)

        pbar.update(1)
        pbar.set_postfix({"página": pagina, "salvos_total": total_saved})

        pagina += 1
        if total_paginas and pagina > total_paginas:
            break

        time.sleep(sleep_between)

    pbar.close()
    print(f"✅ Concluído: {total_saved} linhas salvas em {save_path}")
    return total_saved

# Loop para vários anos
for ano in range(2020, 2025):
    data_inicial = f"{ano}-01-01"
    data_final = f"{ano}-12-31"
    caminho_licitacoes = BASE_DIR / f"licitacoes_{ano}.csv"

    print(f"\n Extraindo licitações do ano {ano}...")
    extrair_licitacoes(data_inicial, data_final, caminho_licitacoes)
    print(f"📁 Arquivo salvo: {caminho_licitacoes}\n")

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import time
from pathlib import Path

# Defina a pasta base onde os arquivos serão salvos
BASE_DIR = Path.home() / "/Users/nathaliasales/Dropbox/MiDES-data-paper-replication/Data/Raw/comprasdados"
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Função para requisição com retry
def requisicao_com_retry(url, params, tentativas=3, espera=5):
    for i in range(tentativas):
        resp = requests.get(url, params=params)
        if resp.status_code == 200:
            return resp
        elif resp.status_code == 429:
            print(f"Recebido 429, esperando {espera} segundos... (tentativa {i+1})")
            time.sleep(espera)
        else:
            print(f"Erro na requisição: {resp.status_code}")
            print("Conteúdo da resposta:", resp.text[:500])
            return None
    print("Falha após múltiplas tentativas.")
    return None

# Função para extrair compras sem licitação
def extrair_compras_sem_licitacao(ano, save_path, page_size=500, sleep_between=1):
    url_base = "https://dadosabertos.compras.gov.br/modulo-legado/5_consultarComprasSemLicitacao"
    pagina = 1
    total_paginas = None
    total_saved = 0
    first_write = True

    pbar = tqdm(desc=f"Compras sem licitação {ano}", unit="pág", dynamic_ncols=True)

    while True:
        params = {
            "pagina": pagina,
            "tamanhoPagina": page_size,
            "dt_ano_aviso": ano
        }

        resp = requisicao_com_retry(url_base, params)
        if not resp:
            break

        try:
            dados = resp.json()
        except ValueError:
            print("Resposta não é JSON — fim da extração.")
            break

        if isinstance(dados, dict) and "resultado" in dados:
            resultados = dados.get("resultado", [])
            if total_paginas is None:
                total_paginas = dados.get("totalPaginas", 1)
                print(f"🔎 Total de páginas esperado: {total_paginas}")
        else:
            resultados = dados

        if not resultados:
            print(f"Nenhum dado novo na página {pagina} — encerrando loop.")
            break

        df = pd.DataFrame(resultados)
        if not df.empty:
            df.to_csv(save_path, mode="w" if first_write else "a",
                      header=first_write, index=False, encoding="utf-8-sig")
            first_write = False
            total_saved += len(df)

        pbar.update(1)
        pbar.set_postfix({"página": pagina, "salvos_total": total_saved})

        pagina += 1
        if total_paginas and pagina > total_paginas:
            break

        time.sleep(sleep_between)

    pbar.close()
    print(f"✅ Concluído: {total_saved} linhas salvas em {save_path}")
    return total_saved

# 🔹 Loop para vários anos
for ano in range(2020, 2025):
    
    caminho_compras = BASE_DIR / f"compras_sem_licitacao_{ano}.csv"

    print(f"\n Extraindo compras do ano {ano}...")
    extrair_compras_sem_licitacao(ano, caminho_compras)
    print(f"📁 Arquivo salvo: {caminho_compras}\n")

In [ ]:
import requests
import pandas as pd
from pathlib import Path
import time

# Pasta para salvar os arquivos
BASE_DIR = Path.home() / "/Users/nathaliasales/Dropbox/MiDES-data-paper-replication/Data/Raw/comprasdados"
BASE_DIR.mkdir(parents=True, exist_ok=True)

def baixar_json_paginas(url_base, params=None, save_path=None, tentativas=3, espera=5):
    """
    Baixa todas as páginas de um endpoint JSON paginado e salva em CSV.
    """
    if params is None:
        params = {}
    pagina = 1
    resultados_totais = []

    while True:
        params["pagina"] = pagina
        sucesso = False

        for i in range(tentativas):
            resp = requests.get(url_base, params=params)
            if resp.status_code == 200:
                sucesso = True
                break
            else:
                print(f"Erro {resp.status_code}. Tentativa {i+1}/{tentativas}. Esperando {espera}s...")
                time.sleep(espera)
        if not sucesso:
            print(f"❌ Falha ao baixar página {pagina}. Encerrando.")
            break

        try:
            dados = resp.json()
        except ValueError:
            print("❌ Resposta não é JSON válido.")
            break

        resultados = dados.get("resultado", [])
        if not resultados:
            print(f"✅ Nenhum dado nesta página ({pagina}). Concluído.")
            break

        resultados_totais.extend(resultados)
        print(f"📄 Página {pagina} baixada | Total registros até agora: {len(resultados_totais)}")

        if dados.get("paginasRestantes", 0) == 0:
            break

        pagina += 1
        time.sleep(1)  # evita sobrecarga do servidor

    if save_path and resultados_totais:
        df = pd.DataFrame(resultados_totais)
        df.to_csv(save_path, index=False, encoding="utf-8-sig")
        print(f"✅ CSV salvo em {save_path} | Total de registros: {len(resultados_totais)}")

    return resultados_totais

# UASG
url_uasg = "https://dadosabertos.compras.gov.br/modulo-uasg/1_consultarUasg"
caminho_uasg = BASE_DIR / "uasg.csv"
params_uasg = {
    "statusUasg": "true"
}

print("\n🔹 Iniciando download do dataset UASG...")
baixar_json_paginas(url_uasg, params=params_uasg, save_path=caminho_uasg)

# Órgãos
url_orgao = "https://dadosabertos.compras.gov.br/modulo-uasg/2_consultarOrgao"
caminho_orgao = BASE_DIR / "orgaos.csv"
params_orgao = {
    "statusOrgao": "true"
}

print("\n🔹 Iniciando download do dataset Órgãos...")
baixar_json_paginas(url_orgao, params=params_orgao, save_path=caminho_orgao)